In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages

from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver


# ----------------------------
# State
# ----------------------------
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# ----------------------------
# LLM
# ----------------------------
llm = ChatGroq( model="llama-3.3-70b-versatile" )


# ----------------------------
# Node
# ----------------------------
def llm_response(state: ChatState):
    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }

checkpointer=MemorySaver()

# ----------------------------
# Build graph
# ----------------------------
graph = StateGraph(ChatState)

graph.add_node("llm_response", llm_response)

graph.add_edge(START, "llm_response")
graph.add_edge("llm_response", END)


# Compile workflow
workflow = graph.compile(checkpointer=checkpointer)


# ----------------------------
# Interactive chat loop
# ----------------------------
state = {"messages": []}

print("LangGraph + Groq Chatbot")
print("Type 'quit' to exit.\n")

thread_id='1'

while True:
    user_input = input("You: ")

    if user_input.lower() in {"quit", "exit", "q"}:
        print("Goodbye!")
        break

    # Add user message
    # state["messages"].append(HumanMessage(content=user_input))
    # print(state['messages'])
    # Invoke workflow

    config = {
    'configurable': {
        'thread_id': thread_id
    }
}

    response = workflow.invoke(
        {
            'messages': [HumanMessage(content=user_input)]
        },
        config=config
    )

    # Get the latest AI message only
    ai_message = response['messages'][-1]

    print("AI:", ai_message.content)
    print()

LangGraph + Groq Chatbot
Type 'quit' to exit.

AI: It seems like you didn't type anything. Please go ahead and ask your question or share what's on your mind, and I'll do my best to help.

AI: India, officially known as the Republic of India, is a country located in South Asia. It is the seventh-largest country by land area, the second-most populous country with over 1.38 billion people, and the most populous democracy in the world.

Here are some key facts about India:

1. **Geography**: India is bounded by the Indian Ocean to the south, the Arabian Sea to the west, and the Bay of Bengal to the east. It shares borders with several countries, including Pakistan, China, Nepal, Bhutan, Bangladesh, and Myanmar.
2. **Culture**: India is a diverse country with a rich cultural heritage. It is home to many languages, including Hindi, English, and over 20 other recognized languages. The country has a long history of philosophy, art, architecture, music, and dance.
3. **History**: India has a l

In [ ]:
[HumanMessage(content='what is capital of india', additional_kwargs={}, response_metadata={}, id='699000a0-530f-4e9d-9d94-002c68345b08'), AIMessage(content='The capital of India is **New Delhi**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 40, 'total_tokens': 51, 'completion_time': 0.022579299, 'completion_tokens_details': None, 'prompt_time': 0.001224661, 'prompt_tokens_details': None, 'queue_time': 0.051648168, 'total_time': 0.02380396}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdd43-772e-74d1-97a9-6f32a3ceda81-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 11, 'total_tokens': 51})]